# Aula 05 - Versionamento de Dados e DVC
### Laboratório Prático

**Disciplina:** MLOps

---

## Recapitulação

Pessoal, para chegar até aqui a gente já passou por:

| Aula | O que fizemos |
|---|---|
| 03 | Servimos um modelo pré-treinado em produção, com validação de entrada |
| 04 | Montamos um pipeline estruturado: Extract, Validate, Transform, Train, Evaluate, Save |
| **05** | **Versionar dados e pipelines com DVC, para saber com quais dados cada modelo foi treinado** |

---

## O problema que vamos resolver

Imaginem este cenário:

> O modelo de vocês estava com 95% de acurácia há 2 semanas. Hoje está com 78%.
> Alguém atualizou o dataset, mas ninguém sabe qual versão dos dados gerou o modelo bom.

Com Git puro nós versionamos o código, mas um `dataset.csv` de 2GB não vai para o Git.
É exatamente esse problema que o DVC resolve.


---

## Como este laboratório funciona

| Fase | Dataset | O que você faz |
|---|---|---|
| **1. Observe** | Iris | Execute o setup e veja um repo DVC já configurado |
| **2. Entenda** | Iris | Leia os arquivos gerados e responda as perguntas |
| **3. Faça** | Wine | Crie do zero: git init, dvc init, versionar, rollback |
| **4. Pipeline** | Wine | Defina um pipeline reprodutível com `dvc.yaml` |

---

## Pré-requisito - verifique antes de começar

In [1]:
import subprocess

def verificar(cmd, nome):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    versao = (r.stdout + r.stderr).strip().split("\n")[0]
    status = "OK" if r.returncode == 0 else "NÃO ENCONTRADO"
    print(f"{status}  {nome}: {versao}")

verificar("git --version",  "Git")
verificar("dvc --version",  "DVC")

print()
print("Se DVC não estiver instalado:  pip install dvc")

OK  Git: git version 2.56.0
OK  DVC: 3.67.1

Se DVC não estiver instalado:  pip install dvc


In [2]:
import os, json, shutil
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, load_wine

# Helper: executa um comando shell em um diretório e exibe o output
def run(cmd, cwd=".", echo=True):
    if echo:
        print(f"$ {cmd}")
    r = subprocess.run(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace"
    )
    output = r.stdout.strip()
    if output:
        print(output)
    return output

print("Imports OK")

Imports OK


---
# FASE 1 - OBSERVE
### Dataset: Iris | Repositório já configurado

A célula abaixo eu já deixei pronta para vocês. Ela cria um repositório DVC completo com duas versões do dataset Iris:
- **v1:** dataset completo (150 amostras)
- **v2:** apenas as classes *setosa* e *versicolor* (100 amostras)

Vocês não precisam entender cada linha agora, apenas executem e observem o que é gerado. Na Fase 2 a gente explica cada arquivo.

In [3]:
# Cria o demo_iris/ com duas versões do dataset 
DEMO = "demo_iris"

# Limpa e recria o diretório
if os.path.exists(DEMO):
    shutil.rmtree(DEMO)
os.makedirs(f"{DEMO}/data")
os.makedirs(f"{DEMO}/remote_storage")

# Configura Git
run("git init",                              cwd=DEMO)
run('git config user.email "rafaelaeduardadeoliveira951@gmail.com"', cwd=DEMO)
run('git config user.name "Rafaela091"',   cwd=DEMO)

# Inicializa DVC
run("dvc init",                              cwd=DEMO)
run("git add -A && git commit -m 'init: git + dvc'", cwd=DEMO)

# Configura remote local
run("dvc remote add -d local_remote remote_storage", cwd=DEMO)
with open(f"{DEMO}/.gitignore", "w") as f:
    f.write("remote_storage/\n")
run("git add .dvc/config .gitignore && git commit -m 'config: remote local'", cwd=DEMO)

print()
print("COMMIT 1: v1 - dataset completo (150 amostras)")
iris = load_iris()
df_v1 = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df_v1["target"]      = iris.target
df_v1["target_name"] = df_v1["target"].map(dict(enumerate(iris.target_names)))
df_v1.to_csv(f"{DEMO}/data/iris.csv", index=False)
print(f"Dataset v1: {len(df_v1)} amostras, {df_v1['target_name'].value_counts().to_dict()}")

run("dvc add data/iris.csv",                             cwd=DEMO)
run("git add data/iris.csv.dvc data/.gitignore",         cwd=DEMO)
run('git commit -m "data: adiciona iris.csv v1 (150 amostras)"', cwd=DEMO)
run("dvc push",                                          cwd=DEMO)

print()
print("COMMIT 2: v2 - apenas setosa e versicolor (100 amostras)")
df_v2 = df_v1[df_v1["target"].isin([0, 1])].copy()
df_v2.to_csv(f"{DEMO}/data/iris.csv", index=False)
print(f"Dataset v2: {len(df_v2)} amostras, {df_v2['target_name'].value_counts().to_dict()}")

run("dvc add data/iris.csv",                             cwd=DEMO)
run("git add data/iris.csv.dvc",                         cwd=DEMO)
run('git commit -m "data: filtra iris.csv v2 (100 amostras, 2 classes)"', cwd=DEMO)
run("dvc push",                                          cwd=DEMO)

print()
print("Demo pronto! Repositório com 2 versões do dataset Iris.")

$ git init
Initialized empty Git repository in /var/home/isaacnilsonv/Downloads/demo_iris/.git/
$ git config user.email "rafaelaeduardadeoliveira951@gmail.com"
$ git config user.name "Rafaela091"
$ dvc init


Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>
$ git add -A && git commit -m 'init: git + dvc'
[main (root-commit) 3514125] init: git + dvc
 3 files changed, 6 insertions(+)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore
$ dvc remote a

Setting 'local_remote' as a default remote.
$ git add .dvc/config .gitignore && git commit -m 'config: remote local'
[main ec1f378] config: remote local
 2 files changed, 5 insertions(+)
 create mode 100644 .gitignore

COMMIT 1: v1 - dataset completo (150 amostras)
Dataset v1: 150 amostras, {np.str_('setosa'): 50, np.str_('versicolor'): 50, np.str_('virginica'): 50}
$ dvc add data/iris.csv


⠋ Checking graph

To track the changes with git, run:

	git add data/.gitignore data/iris.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
$ git add data/iris.csv.dvc data/.gitignore
$ git commit -m "data: adiciona iris.csv v1 (150 amostras)"
[main 019b0fc] data: adiciona iris.csv v1 (150 amostras)
 2 files changed, 6 insertions(+)
 create mode 100644 data/.gitignore
 create mode 100644 data/iris.csv.dvc
$ dvc push


1 file pushed

COMMIT 2: v2 - apenas setosa e versicolor (100 amostras)
Dataset v2: 100 amostras, {np.str_('setosa'): 50, np.str_('versicolor'): 50}
$ dvc add data/iris.csv


⠋ Checking graph

To track the changes with git, run:

	git add data/iris.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
$ git add data/iris.csv.dvc
$ git commit -m "data: filtra iris.csv v2 (100 amostras, 2 classes)"
[main 57df261] data: filtra iris.csv v2 (100 amostras, 2 classes)
 1 file changed, 2 insertions(+), 2 deletions(-)
$ dvc push


1 file pushed

Demo pronto! Repositório com 2 versões do dataset Iris.


In [4]:
# Observe a estrutura do repositório 
print("=" * 50)
print("ESTRUTURA DO REPOSITÓRIO demo_iris/")
print("=" * 50)
for root, dirs, files in os.walk(DEMO):
    # Ignora .git e cache
    dirs[:] = [d for d in dirs if d not in [".git", "cache", "tmp", "remote_storage"]]
    nivel   = root.replace(DEMO, "").count(os.sep)
    indent  = "  " * nivel
    nome    = os.path.basename(root)
    if nivel > 0:
        print(f"{indent}{nome}/")
    for f in files:
        print(f"{indent}  {f}")

ESTRUTURA DO REPOSITÓRIO demo_iris/
  .dvcignore
  .gitignore
  data/
    .gitignore
    iris.csv.dvc
    iris.csv
  .dvc/
    config
    .gitignore


In [5]:
# Veja o histórico de commits 
print("=" * 50)
print("HISTÓRICO GIT (git log)")
print("=" * 50)
run("git log --oneline", cwd=DEMO)

HISTÓRICO GIT (git log)
$ git log --oneline
57df261 data: filtra iris.csv v2 (100 amostras, 2 classes)
019b0fc data: adiciona iris.csv v1 (150 amostras)
ec1f378 config: remote local
3514125 init: git + dvc


'57df261 data: filtra iris.csv v2 (100 amostras, 2 classes)\n019b0fc data: adiciona iris.csv v1 (150 amostras)\nec1f378 config: remote local\n3514125 init: git + dvc'

In [6]:
# Leia o arquivo .dvc (versão atual) 
print("=" * 50)
print("CONTEÚDO DE data/iris.csv.dvc")
print("=" * 50)
with open(f"{DEMO}/data/iris.csv.dvc") as f:
    print(f.read())

print("=" * 50)
print("CONTEÚDO DE data/.gitignore")
print("=" * 50)
with open(f"{DEMO}/data/.gitignore") as f:
    print(f.read())

CONTEÚDO DE data/iris.csv.dvc
outs:
- md5: 528b6dac57633828444b45aa8e50b471
  size: 2789
  hash: md5
  path: iris.csv

CONTEÚDO DE data/.gitignore
/iris.csv



In [7]:
# Leia o dataset na versão atual (v2) 
df_atual = pd.read_csv(f"{DEMO}/data/iris.csv")
print(f"Dataset ATUAL: {len(df_atual)} amostras")
print(df_atual["target_name"].value_counts())

Dataset ATUAL: 100 amostras
target_name
setosa        50
versicolor    50
Name: count, dtype: int64


In [8]:
# Volte para a versão anterior (v1) 
print("Voltando para o commit anterior (HEAD~1 = v1)...")
run("git checkout HEAD~1 -- data/iris.csv.dvc", cwd=DEMO)
run("dvc checkout",                             cwd=DEMO)

print()
df_v1_restaurado = pd.read_csv(f"{DEMO}/data/iris.csv")
print(f"Dataset RESTAURADO: {len(df_v1_restaurado)} amostras")
print(df_v1_restaurado["target_name"].value_counts())

# Volta para a versão mais recente
run("git checkout HEAD -- data/iris.csv.dvc", cwd=DEMO)
run("dvc checkout",                           cwd=DEMO)
print()
print("(voltou para v2)")

Voltando para o commit anterior (HEAD~1 = v1)...
$ git checkout HEAD~1 -- data/iris.csv.dvc
$ dvc checkout


M       data/iris.csv

Dataset RESTAURADO: 150 amostras
target_name
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64
$ git checkout HEAD -- data/iris.csv.dvc
$ dvc checkout


M       data/iris.csv

(voltou para v2)


---
# FASE 2 - ENTENDA
### Dataset: Iris | Leia os arquivos e responda

Você acabou de ver o DVC em ação. Agora vamos entender **o que cada peça significa**.

### O arquivo `.dvc` - o coração do DVC

O arquivo `data/iris.csv.dvc` registra o hash MD5, o tamanho e o caminho do dataset. O hash observado na execução identifica uma versão específica dos bytes do arquivo.

**Pergunta 2.1** - O arquivo `.dvc` é pequeno (alguns bytes) e vai para o Git. O `iris.csv` real (com os dados) vai para o DVC cache. Por que essa separação é inteligente?

*Resposta:* O Git guarda apenas o ponteiro pequeno e o histórico de código, sem carregar uma cópia integral do CSV em cada commit. O DVC guarda o conteúdo pelo hash no cache e o envia ao remote. Assim, o commit identifica exatamente os dados usados e outra pessoa pode recuperá-los com `dvc pull`, desde que tenha acesso ao remote.

**Pergunta 2.2** - O hash MD5 muda sempre que o conteúdo do arquivo muda. Se dois datasets tiverem o mesmo hash, o que isso significa?

*Resposta:* No contexto do DVC, indica conteúdo byte a byte idêntico e permite reutilizar o mesmo objeto do cache, sem duplicar o armazenamento. Hash igual não quer dizer apenas mesmas linhas em ordem diferente: nesse caso os bytes mudam. MD5 não é prova criptográfica absoluta de identidade, pois colisões são possíveis; para rastreamento de versões do laboratório, é o identificador usado pelo DVC.


In [9]:
# Execute e observe: o iris.csv está no .gitignore - o Git não o rastreia
print("Arquivos rastreados pelo Git:")
run("git ls-files", cwd=DEMO)

print()
print("Status do DVC:")
run("dvc status", cwd=DEMO)

Arquivos rastreados pelo Git:
$ git ls-files
.dvc/.gitignore
.dvc/config
.dvcignore
.gitignore
data/.gitignore
data/iris.csv.dvc

Status do DVC:
$ dvc status


Data and pipelines are up to date.


'Data and pipelines are up to date.'

**Pergunta 2.3** - Na saída de `git ls-files`, o `iris.csv` aparece? E o `iris.csv.dvc`? O que isso nos diz?

*Resposta:* `data/iris.csv` não aparece; `data/iris.csv.dvc` aparece. O CSV é ignorado pelo Git por `data/.gitignore`; o ponteiro `.dvc` é versionado pelo Git e conecta o commit ao conteúdo guardado pelo DVC.

**Pergunta 2.4** - O que significa a saída `Data and pipelines are up to date.` do `dvc status`?

*Resposta:* O conteúdo do workspace corresponde aos hashes registrados nos arquivos DVC e no lockfile para a versão atual, sem alterações pendentes que exijam atualizar dados ou executar stages. Isso não prova que o remote esteja sincronizado; `dvc status -c` e `dvc push` verificam/sincronizam o remoto.


In [10]:
# Observe o cache do DVC - onde os dados reais ficam armazenados
cache_dir = os.path.join(DEMO, ".dvc", "cache", "files", "md5")
if os.path.exists(cache_dir):
    print("Arquivos no cache DVC (organizados por hash):")
    for root, _, files in os.walk(cache_dir):
        for f in files:
            full = os.path.join(root, f)
            size = os.path.getsize(full)
            nome = os.path.relpath(full, cache_dir)
            print(f"  {nome}  ({size} bytes)")
else:
    # versões antigas do DVC usam estrutura diferente
    cache_dir2 = os.path.join(DEMO, ".dvc", "cache")
    for root, dirs, files in os.walk(cache_dir2):
        dirs[:] = [d for d in dirs if d != "tmp"]
        for f in files:
            full = os.path.join(root, f)
            print(f"  {os.path.relpath(full, cache_dir2)}  ({os.path.getsize(full)} bytes)")

Arquivos no cache DVC (organizados por hash):
  9b/81a5b2beab90b5e381dfb33627d536  (4189 bytes)
  52/8b6dac57633828444b45aa8e50b471  (2789 bytes)


**Pergunta 2.5** - O cache contém **dois** arquivos (um para v1 e outro para v2). Por que o DVC armazena ambas as versões e não descarta a antiga?

*Resposta:* Cada hash corresponde a um conteúdo distinto. Manter os dois objetos permite restaurar v1 ou v2 sem recalcular o dataset, reproduzir experimentos e auditar qual versão foi usada. `dvc gc` pode remover objetos não necessários; por isso o remote deve preservar versões que ainda serão consultadas.

**Pergunta 2.6** - Quando você executou `git checkout HEAD~1 -- data/iris.csv.dvc` seguido de `dvc checkout`, o que cada comando fez exatamente?

*Resposta:* O primeiro trouxe do commit anterior o ponteiro `.dvc` de v1 para o índice e o diretório de trabalho do Git. Ele não alterou o CSV real. O segundo leu o hash desse ponteiro e materializou o CSV de v1 a partir do cache do DVC. Para voltar à v2, o notebook restaura o ponteiro de `HEAD` e executa `dvc checkout` novamente. Se faltasse o objeto no cache, seria necessário `dvc pull` do remote.


---
# FASE 3 - FAÇA
### Dataset: Wine | Crie um repositório DVC do zero

Agora é a vez de vocês. Vocês vão criar um repositório Git + DVC do zero para o Wine Dataset e gerar duas versões dos dados.

Sigam as instruções em cada célula. Onde houver `# TODO`, vocês devem completar.

**Versão 1 (v1):** dataset Wine completo (178 amostras, 3 classes)
**Versão 2 (v2):** apenas as classes 0 e 1 (130 amostras)

In [11]:
# Passo 1: Crie a estrutura de diretórios
WINE = "workspace_wine"

if os.path.exists(WINE):
    shutil.rmtree(WINE)
os.makedirs(f"{WINE}/data", exist_ok=True)
os.makedirs(f"{WINE}/remote_storage", exist_ok=True)
os.makedirs(f"{WINE}/src", exist_ok=True)
print(f"Pasta {WINE}/ criada")


Pasta workspace_wine/ criada


In [12]:
# Passo 2: Inicialize o Git com a identidade pública do GitHub indicado
run("git init", cwd=WINE)
run('git config user.email "rafaelaeduardadeoliveira951@gmail.com"', cwd=WINE)
run('git config user.name "Rafaela091"', cwd=WINE)
print("Git inicializado")


$ git init
Initialized empty Git repository in /var/home/isaacnilsonv/Downloads/workspace_wine/.git/
$ git config user.email "rafaelaeduardadeoliveira951@gmail.com"
$ git config user.name "Rafaela091"
Git inicializado


In [13]:
# Passo 3: Inicialize o DVC e versione seus arquivos de configuração
run("dvc init", cwd=WINE)
run("git add -A && git commit -m 'init: git + dvc'", cwd=WINE)
print("DVC inicializado")


$ dvc init


Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>
$ git add -A && git commit -m 'init: git + dvc'
[main (root-commit) f11ad93] init: git + dvc
 3 files changed, 6 insertions(+)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore
DVC inicializa

In [14]:
# Passo 4: Configure um remote local. O conteúdo do remote não entra no Git.
run("dvc remote add -d local_remote remote_storage", cwd=WINE)
with open(f"{WINE}/.gitignore", "w") as f:
    f.write("remote_storage/\n")
run("git add .dvc/config .gitignore", cwd=WINE)
run("git commit -m 'config: remote local'", cwd=WINE)
print("Remote local configurado em remote_storage/")


$ dvc remote add -d local_remote remote_storage


Setting 'local_remote' as a default remote.
$ git add .dvc/config .gitignore
$ git commit -m 'config: remote local'
[main 4c06e22] config: remote local
 2 files changed, 5 insertions(+)
 create mode 100644 .gitignore
Remote local configurado em remote_storage/


In [15]:
# Passo 5: Crie e rastreie a versão 1 do dataset 

# Gera o dataset Wine completo
wine = load_wine()
df_wine_v1 = pd.DataFrame(data=wine.data, columns=wine.feature_names)
df_wine_v1["target"]      = wine.target
df_wine_v1["target_name"] = df_wine_v1["target"].map(dict(enumerate(wine.target_names)))
df_wine_v1.to_csv(f"{WINE}/data/wine.csv", index=False)

print(f"Dataset v1 criado: {len(df_wine_v1)} amostras")
print(df_wine_v1["target_name"].value_counts())
print()

# Passo 8 concluído: rastreie o arquivo com DVC
# Comando: dvc add data/wine.csv
run("dvc add data/wine.csv", cwd=WINE)

# Passo 9 concluído: adicione os arquivos gerados pelo DVC ao Git e faça commit
# Os arquivos a adicionar são: data/wine.csv.dvc e data/.gitignore
run("git add data/wine.csv.dvc data/.gitignore", cwd=WINE)
run('git commit -m "data: wine.csv v1 (178 amostras, 3 classes)"', cwd=WINE)

# Passo 10 concluído: envie os dados para o remote
run("dvc push", cwd=WINE)

print("Versão 1 commitada!")

Dataset v1 criado: 178 amostras
target_name
class_1    71
class_0    59
class_2    48
Name: count, dtype: int64

$ dvc add data/wine.csv


⠋ Checking graph

To track the changes with git, run:

	git add data/wine.csv.dvc data/.gitignore

To enable auto staging, run:

	dvc config core.autostage true
$ git add data/wine.csv.dvc data/.gitignore
$ git commit -m "data: wine.csv v1 (178 amostras, 3 classes)"
[main 4d8f78f] data: wine.csv v1 (178 amostras, 3 classes)
 2 files changed, 6 insertions(+)
 create mode 100644 data/.gitignore
 create mode 100644 data/wine.csv.dvc
$ dvc push


1 file pushed
Versão 1 commitada!


In [16]:
# Passo 6: Modifique o dataset e crie a versão 2 

# Passo 11 concluído: filtre apenas as classes 0 e 1 do df_wine_v1
# e salve em workspace_wine/data/wine.csv (sobrescrevendo)
df_wine_v2 = df_wine_v1[df_wine_v1["target"].isin([0, 1])].copy()
df_wine_v2.to_csv(f"{WINE}/data/wine.csv", index=False)

print(f"Dataset v2: {len(df_wine_v2)} amostras")
print(df_wine_v2["target_name"].value_counts())
print()

# Passo 12 concluído: rastreie a nova versão com DVC
run("dvc add data/wine.csv", cwd=WINE)

# Passo 13 concluído: commit e push da versão 2
run("git add data/wine.csv.dvc", cwd=WINE)
run('git commit -m "data: wine.csv v2 (130 amostras, 2 classes)"', cwd=WINE)
run("dvc push", cwd=WINE)

print("Versão 2 commitada!")

Dataset v2: 130 amostras
target_name
class_1    71
class_0    59
Name: count, dtype: int64

$ dvc add data/wine.csv


⠋ Checking graph

To track the changes with git, run:

	git add data/wine.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
$ git add data/wine.csv.dvc
$ git commit -m "data: wine.csv v2 (130 amostras, 2 classes)"
[main c4fd424] data: wine.csv v2 (130 amostras, 2 classes)
 1 file changed, 2 insertions(+), 2 deletions(-)
$ dvc push


1 file pushed
Versão 2 commitada!


In [17]:
# Passo 7: Veja o histórico 
print("Histórico de commits:")
run("git log --oneline", cwd=WINE)

Histórico de commits:
$ git log --oneline
c4fd424 data: wine.csv v2 (130 amostras, 2 classes)
4d8f78f data: wine.csv v1 (178 amostras, 3 classes)
4c06e22 config: remote local
f11ad93 init: git + dvc


'c4fd424 data: wine.csv v2 (130 amostras, 2 classes)\n4d8f78f data: wine.csv v1 (178 amostras, 3 classes)\n4c06e22 config: remote local\nf11ad93 init: git + dvc'

In [18]:
# Passo 8: O momento uau - volte para a versão 1 

print("Dataset ATUAL (v2):")
df_check = pd.read_csv(f"{WINE}/data/wine.csv")
print(f"  {len(df_check)} amostras | classes: {sorted(df_check['target'].unique())}")
print()

# Passo 14 concluído: use git checkout para restaurar o .dvc da versão anterior
# Comando: git checkout HEAD~1 -- data/wine.csv.dvc
run("git checkout HEAD~1 -- data/wine.csv.dvc", cwd=WINE)

# Passo 15 concluído: use dvc checkout para restaurar os DADOS reais da versão anterior
run("dvc checkout", cwd=WINE)

print("Dataset RESTAURADO (v1):")
df_check = pd.read_csv(f"{WINE}/data/wine.csv")
print(f"  {len(df_check)} amostras | classes: {sorted(df_check['target'].unique())}")
print()

# Volta para v2 para continuar o lab
run("git checkout HEAD -- data/wine.csv.dvc", cwd=WINE)
run("dvc checkout",                          cwd=WINE)
print("(voltou para v2)")

# Comprova que o remote também permite recuperar a versão corrente sem o cache local.
os.remove(f"{WINE}/data/wine.csv")
run("dvc pull", cwd=WINE)
df_pulled = pd.read_csv(f"{WINE}/data/wine.csv")
assert len(df_pulled) == 130
print(f"dvc pull restaurou v2: {len(df_pulled)} amostras")

Dataset ATUAL (v2):
  130 amostras | classes: [np.int64(0), np.int64(1)]

$ git checkout HEAD~1 -- data/wine.csv.dvc
$ dvc checkout


M       data/wine.csv
Dataset RESTAURADO (v1):
  178 amostras | classes: [np.int64(0), np.int64(1), np.int64(2)]

$ git checkout HEAD -- data/wine.csv.dvc
$ dvc checkout


M       data/wine.csv
(voltou para v2)
$ dvc pull


A       data/wine.csv
1 file added
dvc pull restaurou v2: 130 amostras


### Responda - O que você aprendeu?

**Pergunta 3.1** - Explique com suas palavras o papel de cada comando no fluxo abaixo:

- `dvc add data/wine.csv`: calcula o hash, guarda o conteúdo no cache e escreve/atualiza `data/wine.csv.dvc` e a regra de ignore.
- `git add data/wine.csv.dvc`: coloca o ponteiro no índice do Git; na v1 também é preciso adicionar `data/.gitignore`.
- `git commit`: grava no histórico o ponteiro (e os outros arquivos adicionados), associando o commit à versão dos dados.
- `dvc push`: copia os objetos do cache para o remote configurado, para que possam ser recuperados em outro workspace ou depois da perda do cache local.

**Pergunta 3.2** - Por que o rollback exigiu **dois** comandos (`git checkout` + `dvc checkout`) em vez de apenas um?

*Resposta:* O Git restaura apenas o ponteiro, porque o CSV está ignorado. O DVC usa o hash do ponteiro restaurado para substituir o CSV no diretório de trabalho pelo conteúdo da versão escolhida. O notebook verificou a volta de 130 para 178 amostras e depois para 130.

**Pergunta 3.3** - Em produção, o DVC remote seria uma pasta local? O que usariam em vez disso e por quê?

*Resposta:* Normalmente seria um armazenamento compartilhado e durável, como S3, Google Cloud Storage, Azure Blob ou um servidor SSH, com credenciais e controle de acesso. Isso permite que a equipe e a automação façam `dvc push/pull` em máquinas diferentes e preserva os objetos mesmo que o disco de uma pessoa falhe. A pasta local serve apenas para demonstrar o mecanismo.


---
# FASE 4 - PIPELINE
### Dataset: Wine | Pipeline reprodutível com `dvc.yaml`

Na Aula 04 você construiu um pipeline de ML em Python puro.
Agora vamos converter esse pipeline em **stages DVC**, que são:
- **Reprodutíveis**: `dvc repro` re-executa apenas o que mudou
- **Rastreáveis**: tudo vai para o `dvc.lock` com hashes
- **Versionados**: junto com os dados e o código no Git

### Estrutura que vamos criar

```
workspace_wine/
├ data/wine.csv          ← rastreado pelo DVC
├ src/
│  ├ prepare.py          ← stage 1: split treino/teste
│  └ train.py            ← stage 2: treina e avalia
├ params.yaml            ← hiperparâmetros versionados
├ dvc.yaml               ← definição do pipeline
└ artifacts/
   ├ model.joblib        ← saída rastreada pelo DVC
   └ metrics.json        ← métricas rastreadas pelo DVC
```

In [19]:
%%writefile workspace_wine/params.yaml
prepare:
  test_size: 0.2
  random_state: 42

train:
  n_estimators: 100
  random_state: 42

Writing workspace_wine/params.yaml


In [20]:
%%writefile workspace_wine/src/prepare.py
"""
Stage 1 - prepare.py
Lê data/wine.csv, faz o split treino/teste e salva em data/prepared/
"""
import os, yaml
import pandas as pd
from sklearn.model_selection import train_test_split

# Carregar parâmetros do params.yaml
with open("params.yaml") as f:
    params = yaml.safe_load(f)["prepare"]

df = pd.read_csv("data/wine.csv")

feature_cols = [c for c in df.columns if c not in ["target", "target_name"]]
X = df[feature_cols]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=params["test_size"],
    random_state=params["random_state"],
    stratify=y
)

os.makedirs("data/prepared", exist_ok=True)
X_train.to_csv("data/prepared/X_train.csv", index=False)
X_test.to_csv("data/prepared/X_test.csv",   index=False)
y_train.to_csv("data/prepared/y_train.csv", index=False)
y_test.to_csv("data/prepared/y_test.csv",   index=False)

print(f"Prepare OK: treino={len(X_train)} | teste={len(X_test)}")

Writing workspace_wine/src/prepare.py


In [21]:
%%writefile workspace_wine/src/train.py
"""
Stage 2 - train.py
Lê data/prepared/, treina o modelo e salva em artifacts/
"""
import os, json, yaml
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import joblib

with open("params.yaml") as f:
    params = yaml.safe_load(f)["train"]

X_train = pd.read_csv("data/prepared/X_train.csv")
X_test  = pd.read_csv("data/prepared/X_test.csv")
y_train = pd.read_csv("data/prepared/y_train.csv").squeeze()
y_test  = pd.read_csv("data/prepared/y_test.csv").squeeze()

model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    RandomForestClassifier(
        n_estimators=params["n_estimators"],
        random_state=params["random_state"]
    ))
])
model.fit(X_train, y_train)

y_pred   = model.predict(X_test)
accuracy = round(float(accuracy_score(y_test, y_pred)), 4)
f1       = round(float(f1_score(y_test, y_pred, average="macro")), 4)

os.makedirs("artifacts", exist_ok=True)
joblib.dump(model, "artifacts/model.joblib")

metrics = {"accuracy": accuracy, "f1_macro": f1}
with open("artifacts/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Train OK: accuracy={accuracy} | f1={f1}")

Writing workspace_wine/src/train.py


In [22]:
# Defina o dvc.yaml 
# O dvc.yaml descreve os stages do pipeline:
#   - cmd:    comando a executar
#   - deps:   arquivos que este stage DEPENDE (se mudarem, o stage re-executa)
#   - params: parâmetros do params.yaml que este stage usa
#   - outs:   arquivos que este stage PRODUZ (rastreados pelo DVC)
#   - metrics: arquivos de métricas (não entram no cache, ficam legíveis)

dvc_yaml_content = """
stages:
  preparar:
    cmd: python src/prepare.py
    deps:
      - data/wine.csv
      - src/prepare.py
    params:
      - params.yaml:
          - prepare.test_size
          - prepare.random_state
    outs:
      - data/prepared/

  treinar:
    cmd: python src/train.py
    deps:
      - data/prepared/
      - src/train.py
    params:
      - params.yaml:
          - train.n_estimators
          - train.random_state
    outs:
      - artifacts/model.joblib
    metrics:
      - artifacts/metrics.json:
          cache: false
"""

with open(f"{WINE}/dvc.yaml", "w") as f:
    f.write(dvc_yaml_content.strip())

print("dvc.yaml criado:")
print(dvc_yaml_content)

dvc.yaml criado:

stages:
  preparar:
    cmd: python src/prepare.py
    deps:
      - data/wine.csv
      - src/prepare.py
    params:
      - params.yaml:
          - prepare.test_size
          - prepare.random_state
    outs:
      - data/prepared/

  treinar:
    cmd: python src/train.py
    deps:
      - data/prepared/
      - src/train.py
    params:
      - params.yaml:
          - train.n_estimators
          - train.random_state
    outs:
      - artifacts/model.joblib
    metrics:
      - artifacts/metrics.json:
          cache: false



In [23]:
# Passo 9: Execute o pipeline pela primeira vez 
os.makedirs(f"{WINE}/src", exist_ok=True)

print("Executando: dvc repro")
print("-" * 50)
run("dvc repro", cwd=WINE)
run("dvc push", cwd=WINE)

Executando: dvc repro
--------------------------------------------------
$ dvc repro


'data/wine.csv.dvc' didn't change, skipping
Running stage 'preparar':
> python src/prepare.py
Prepare OK: treino=104 | teste=26
Generating lock file 'dvc.lock'
Updating lock file 'dvc.lock'

Running stage 'treinar':
> python src/train.py
Train OK: accuracy=1.0 | f1=1.0
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add artifacts/.gitignore data/.gitignore dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.
$ dvc push


6 files pushed


'6 files pushed'

In [24]:
# Veja as métricas 
print("Métricas do pipeline:")
run("dvc metrics show", cwd=WINE)

print()
with open(f"{WINE}/artifacts/metrics.json") as f:
    print(json.dumps(json.load(f), indent=2))

Métricas do pipeline:
$ dvc metrics show


Path                    accuracy    f1_macro
artifacts/metrics.json  1.0         1.0

{
  "accuracy": 1.0,
  "f1_macro": 1.0
}


In [25]:
# Leia o dvc.lock - o registro completo da execução 
print("Conteúdo do dvc.lock:")
with open(f"{WINE}/dvc.lock") as f:
    print(f.read())

Conteúdo do dvc.lock:
schema: '2.0'
stages:
  preparar:
    cmd: python src/prepare.py
    deps:
    - path: data/wine.csv
      hash: md5
      md5: 13696e99470f906fd75a05eec5600141
      size: 10072
    - path: src/prepare.py
      hash: md5
      md5: 66f4fdb4745b177ae6c9a6cf6d79c42f
      size: 948
    params:
      params.yaml:
        prepare.random_state: 42
        prepare.test_size: 0.2
    outs:
    - path: data/prepared/
      hash: md5
      md5: 8e1525883f8df8b272eb1265cbfb492e.dir
      size: 9197
      nfiles: 4
  treinar:
    cmd: python src/train.py
    deps:
    - path: data/prepared/
      hash: md5
      md5: 8e1525883f8df8b272eb1265cbfb492e.dir
      size: 9197
      nfiles: 4
    - path: src/train.py
      hash: md5
      md5: a7195d63bd6d00687858a29ae41a90b5
      size: 1281
    params:
      params.yaml:
        train.n_estimators: 100
        train.random_state: 42
    outs:
    - path: artifacts/metrics.json
      hash: md5
      md5: 4958cbd40c01f674fd4aa264f

In [26]:
# Commit do pipeline 
run("git add dvc.yaml dvc.lock params.yaml src/ data/.gitignore artifacts/.gitignore artifacts/metrics.json", cwd=WINE)
run('git commit -m "pipeline: adiciona stages preparar + treinar"', cwd=WINE)

$ git add dvc.yaml dvc.lock params.yaml src/ data/.gitignore artifacts/.gitignore artifacts/metrics.json
$ git commit -m "pipeline: adiciona stages preparar + treinar"
[main 378537b] pipeline: adiciona stages preparar + treinar
 8 files changed, 161 insertions(+)
 create mode 100644 artifacts/.gitignore
 create mode 100644 artifacts/metrics.json
 create mode 100644 dvc.lock
 create mode 100644 dvc.yaml
 create mode 100644 params.yaml
 create mode 100644 src/prepare.py
 create mode 100644 src/train.py


'[main 378537b] pipeline: adiciona stages preparar + treinar\n 8 files changed, 161 insertions(+)\n create mode 100644 artifacts/.gitignore\n create mode 100644 artifacts/metrics.json\n create mode 100644 dvc.lock\n create mode 100644 dvc.yaml\n create mode 100644 params.yaml\n create mode 100644 src/prepare.py\n create mode 100644 src/train.py'

In [27]:
# Passo 10: Mude um parâmetro e re-execute 
# Passo 16 concluído: abra params.yaml e mude n_estimators de 100 para 200
# Depois execute dvc repro e observe quais stages re-executam

# Modifica o params.yaml programaticamente
import yaml
with open(f"{WINE}/params.yaml") as f:
    params = yaml.safe_load(f)

params["train"]["n_estimators"] = 200  # ← mudança

with open(f"{WINE}/params.yaml", "w") as f:
    yaml.dump(params, f, default_flow_style=False)

print("params.yaml atualizado: n_estimators = 200")
print()
print("Executando: dvc repro")
print("-" * 50)
run("dvc repro", cwd=WINE)
run("dvc push", cwd=WINE)

params.yaml atualizado: n_estimators = 200

Executando: dvc repro
--------------------------------------------------
$ dvc repro


'data/wine.csv.dvc' didn't change, skipping
Stage 'preparar' didn't change, skipping
Running stage 'treinar':
> python src/train.py
Train OK: accuracy=1.0 | f1=1.0
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.
$ dvc push


1 file pushed


'1 file pushed'

In [28]:
# Compare as métricas antes e depois da mudança 
print("Métricas após n_estimators=200:")
run("dvc metrics show", cwd=WINE)

# Commit da nova versão
run("git add params.yaml dvc.lock artifacts/metrics.json", cwd=WINE)
run('git commit -m "exp: aumenta n_estimators para 200"', cwd=WINE)

# Compara métricas entre commits
print()
print("Diferença entre experimentos:")
run("dvc metrics diff HEAD~1", cwd=WINE)
print("O diff fica vazio quando as métricas não mudam. Valores de referência:")
run("git show HEAD~1:artifacts/metrics.json", cwd=WINE)
print("Valores atuais:")
run("git show HEAD:artifacts/metrics.json", cwd=WINE)

Métricas após n_estimators=200:
$ dvc metrics show


Path                    accuracy    f1_macro
artifacts/metrics.json  1.0         1.0
$ git add params.yaml dvc.lock artifacts/metrics.json
$ git commit -m "exp: aumenta n_estimators para 200"
[main 27a211a] exp: aumenta n_estimators para 200
 2 files changed, 5 insertions(+), 6 deletions(-)

Diferença entre experimentos:
$ dvc metrics diff HEAD~1


O diff fica vazio quando as métricas não mudam. Valores de referência:
$ git show HEAD~1:artifacts/metrics.json
{
  "accuracy": 1.0,
  "f1_macro": 1.0
}
Valores atuais:
$ git show HEAD:artifacts/metrics.json
{
  "accuracy": 1.0,
  "f1_macro": 1.0
}


'{\n  "accuracy": 1.0,\n  "f1_macro": 1.0\n}'

### Responda - O que você aprendeu com o pipeline DVC?

**Pergunta 4.1** - Quando você mudou `n_estimators` e executou `dvc repro` novamente, o stage `preparar` re-executou? Por quê ou por que não?

*Resposta:* Não. Só `treinar` depende de `train.n_estimators`; as dependências, parâmetros e saídas de `preparar` permaneceram iguais. O DVC reutilizou os arquivos preparados e treinou de novo com 200 árvores. O `dvc.lock` passou a registrar o novo parâmetro e os hashes das saídas do treino.

**Pergunta 4.2** - Para que serve o arquivo `dvc.lock`? O que ele tem que o `dvc.yaml` não tem?

*Resposta:* `dvc.yaml` declara a receita: comandos, dependências, parâmetros e saídas. `dvc.lock` registra o resultado de uma execução específica, incluindo os valores dos parâmetros usados e os hashes das dependências e saídas. Junto do commit Git e dos objetos no remote, ele permite verificar e recuperar o experimento executado.

**Pergunta 4.3** - Como o `dvc metrics diff` ajuda em um projeto com múltiplos experimentos rodando ao longo de semanas?

*Resposta:* Ele compara as métricas de revisões Git ou experimentos DVC, mostrando qual medida subiu ou caiu após uma mudança. Isso torna a seleção de um modelo mais objetiva e auditável quando vinculada aos respectivos parâmetros, dados e commits. Uma diferença de métrica isolada não prova causalidade ou estabilidade; convém manter o mesmo conjunto de avaliação e analisar a variabilidade.

**Pergunta 4.4** - Conectando com a Aula 04: qual é a diferença entre o `PipelineRunner` que você construiu e os stages do `dvc.yaml`?

*Resposta:* O `PipelineRunner` em Python orquestra as etapas e a lógica de execução dentro do próprio programa. Os stages do DVC declaram, fora do código, as entradas, os parâmetros e as saídas de cada comando. Com isso, `dvc repro` detecta mudanças pelos hashes, executa apenas os stages afetados, registra o estado no `dvc.lock` e integra as saídas ao cache/remote. O código Python continua fazendo a preparação e o treino; o DVC gerencia sua reprodução e proveniência.


---
## Síntese - O que você aprendeu hoje

| Conceito | Comando | Para que serve |
|---|---|---|
| Rastrear dado | `dvc add arquivo` | Cria o `.dvc` (ponteiro) e ignora o arquivo no Git |
| Salvar versão | `git commit` | Salva o ponteiro no histórico do Git |
| Enviar dados | `dvc push` | Envia o dado real para o remote |
| Restaurar dado | `dvc checkout` | Baixa o dado real do cache/remote para o disco |
| Navegar versões | `git checkout <commit>` + `dvc checkout` | Volta dados + código para um ponto no tempo |
| Definir pipeline | `dvc.yaml` | Declara stages, dependências e saídas |
| Reproduzir | `dvc repro` | Executa apenas o que mudou |
| Ver métricas | `dvc metrics show/diff` | Compara resultados entre experimentos |

---

## Próxima aula - Aula 06

**Gestão de Experimentos e Reprodutibilidade com MLflow**

Hoje você versionou dados e parâmetros. Na Aula 06 você vai rastrear os **experimentos completos**, cada combinação de parâmetros, métricas e modelos, com uma interface visual para comparar runs.